In [103]:
import pandas as pd
import numpy as np
import geopandas as gpd
import transbigdata as tbd
import warnings
warnings.filterwarnings('ignore')
from tqdm import tqdm  # 显示循环进度
import time

In [104]:
# 指定要处理的车牌ID
target_vehicle_id = '粤BD00205'

In [105]:
##### 1. 数据读取
# 数据路径
base_path = r'd:/Geo_python/Geo_data/2019/01/20190901'
# 充电站数据
charge_station = pd.read_csv('d:/Geo_python/Geo_data/2019/stations.csv', encoding='gbk')[['lon', 'lat']]
charge_station = charge_station.rename(columns={'lon': 'station_lon'})
charge_station = charge_station.rename(columns={'lat': 'station_lat'})
# 深圳范围数据
sz = gpd.read_file('sz.json')
# 车牌id
id_all = pd.read_csv(r'id_all.csv')
print(f'总车辆数: {len(id_all)}')
print(f'将只处理车牌ID: {target_vehicle_id}')

总车辆数: 21718
将只处理车牌ID: 粤BD00205


In [106]:
# 创建空列表用于存储所有充电事件
all_charge_events = []
# 创建一个字典用于统计每个车辆ID的充电次数
vehicle_charge_count = {}

In [107]:
start = time.perf_counter()  # 开始时间

# 用于存储当前车辆的所有数据
car_data_all = []

# 循环读取每个出租车txt文件，提取当前车辆ID的数据
for i in tqdm(range(1, 289)):
    # 读取文件
    filename = f"{i:03d}.txt"
    full_path = base_path + '_' + filename
    
    # 使用pandas的chunksize参数分块读取大文件，避免内存溢出
    for chunk in pd.read_csv(full_path, header=None, chunksize=100000):
        # 添加列名
        chunk.columns = ['date','date_time','H','id','lon','lat','speed','heading','OpenStatus','shift_type']
        # 筛选当前车辆ID的数据
        car_chunk = chunk[chunk['id'] == target_vehicle_id]
        
        if not car_chunk.empty:
            car_data_all.append(car_chunk)
      
# 如果没有找到该车辆的数据，提示并退出
if not car_data_all:
    print(f"车辆ID {target_vehicle_id} 在2019-09-01没有数据，无法处理")
else:
    # 合并该车辆的所有数据
    data = pd.concat(car_data_all, ignore_index=True)
    
    ###### 1. 数据处理
    ###### 1.1 数据格式整理
    # 删除不需要的列
    data = data.drop(['H','heading','shift_type'], axis=1)
 
    # 转换时间格式
    data['date'] = data['date'].astype(str).str.zfill(8)  # 确保8位日期
    data['date'] = pd.to_datetime(data['date'], format='%Y%m%d').dt.strftime('%Y-%m-%d')
    data['date_time'] = data['date_time'].astype(str).str.zfill(6)  # 确保6位时间
    data['date_time'] = data['date_time'].str.slice(0,2) + ':' + data['date_time'].str.slice(2,4) + ':' + data['date_time'].str.slice(4,6)
    data['time'] = pd.to_datetime(data['date'] + ' ' + data['date_time'])
    
    # 数据预处理
    # 添加前后行的id和载客状态，用于异常点识别
    data['pre_id'] = data['id'].shift(1)
    data['next_id'] = data['id'].shift(-1)
    data['pre_passenger'] = data['OpenStatus'].shift(1)
    data['next_passenger'] = data['OpenStatus'].shift(-1)
    # 剔除异常状态点  前后的载客状态相同但和当前不同，且ID一致
    data = data[-((data['id'] == data['pre_id']) & (data['id'] == data['next_id']) &
                 (data['pre_passenger'] == data['next_passenger']) & (data['OpenStatus'] != data['pre_passenger']))]
    data = data.drop(['pre_id', 'next_id', 'pre_passenger', 'next_passenger'], axis=1)

    ###### 2. GPS在线的识别充电事件（GPS online）
    ###### 2.1 数据冗余剔除,减小数据规模将加快后续的计算效率
    data = tbd.traj_clean_redundant(data, col = ['id','time','lon','lat','speed'])
    # 区域外的数据剔除  增加accuracy参数以包含更多数据点
    data = tbd.clean_outofshape(data, sz, col=['lon', 'lat'], accuracy=1000)
    # 区域内的漂移清洗，以速度、距离、角度三个方法进行清洗
    # data = tbd.traj_clean_drift(data, col=['id', 'time', 'lon', 'lat'], speedlimit=80, dislimit=4000, anglelimit=40)
    
    ###### 2.2 停车与出行识别
    # 定义栅格化参数 
    bounds = [113.75, 22.4, 114.62, 22.86]
    params = tbd.area_to_params(bounds, accuracy=1000) 
    # 识别停车与出行 停车阈值10min(600s)
    stay, move = tbd.traj_stay_move(data, params, col=['id', 'time', 'lon', 'lat'], activitytime=600) # 判断停车阈值10min(600s)
    
    # 如果stay数据为空，跳过在线充电事件识别
    if stay.empty:
        print(f"车辆ID {target_vehicle_id} 筛选位移后的stay数据框为空，跳过在线充电事件识别")
    else:
        # 筛选停车时长大于30分钟的记录
        stay = stay[stay['duration'] >= 1800]  # 30分钟 = 1800秒
        
        if stay.empty:
            print(f"车辆ID {target_vehicle_id} 没有停车时长大于30分钟的记录，跳过在线充电事件识别")
        else:
            # 计算每个停车点到最近充电站的距离
            stay_with_station = tbd.ckdnearest(stay, charge_station, 
                                             Aname=['lon', 'lat'], 
                                             Bname=['station_lon', 'station_lat'])
            
            # 筛选距离充电站200米以内的停车点
            potential_online_charge = stay_with_station[stay_with_station['dist'] <= 200].copy()
            
            if potential_online_charge.empty:
                print(f"车辆ID {target_vehicle_id} 没有距离充电站200米以内的停车点，跳过在线充电事件识别")
            else:
                # 标记为在线充电事件
                potential_online_charge['ischarge'] = 1
                potential_online_charge['charge_type'] = 'online'
                
                # 添加到充电事件列表
                for _, row in potential_online_charge.iterrows():
                    charge_event = {
                        'vehicle_id': row['id'],
                        'start_time': row['stime'],
                        'end_time': row['etime'],
                        'duration': row['duration'],
                        'lon': row['lon'],
                        'lat': row['lat'],
                        'station_distance': row['dist'],
                        'charge_type': 'online'
                    }
                    all_charge_events.append(charge_event)
                
                # 更新车辆充电次数统计
                if target_vehicle_id in vehicle_charge_count:
                    vehicle_charge_count[target_vehicle_id]['online'] = len(potential_online_charge)
                else:
                    vehicle_charge_count[target_vehicle_id] = {'online': len(potential_online_charge), 'offline': 0}
                
                print(f"车辆ID {target_vehicle_id} 识别到 {len(potential_online_charge)} 个在线充电事件")
    
    ###### 3. GPS离线的识别充电事件（GPS offline）
    # 按时间排序
    data_sorted = data.sort_values(by=['id', 'time'])
    
    # 计算时间差
    data_sorted['time_diff'] = data_sorted['time'].diff().dt.total_seconds()
    
    # 标记新的ID开始（时间差为NaN）或时间差大于10分钟的点
    data_sorted['is_offline_end'] = (data_sorted['time_diff'] > 600) | (data_sorted['time_diff'].isna())
    
    # 找出离线结束点的索引
    offline_end_indices = data_sorted[data_sorted['is_offline_end']].index
    
    if len(offline_end_indices) <= 1:  # 需要至少有一个离线结束点（不包括第一个点）
        print(f"车辆ID {target_vehicle_id} 没有识别到离线期间，跳过离线充电事件识别")
    else:
        # 获取离线前的点（离线结束点的前一个点）
        offline_start_indices = [idx-1 for idx in offline_end_indices if idx > 0 and idx-1 in data_sorted.index]
        
        if not offline_start_indices:
            print(f"车辆ID {target_vehicle_id} 的离线期间数据不完整，跳过离线充电事件识别")
        else:
            # 创建离线前后点的数据框
            offline_periods = pd.DataFrame({
                'id': data_sorted.loc[offline_start_indices, 'id'].values,
                'start_time': data_sorted.loc[offline_start_indices, 'time'].values,
                'end_time': data_sorted.loc[offline_end_indices[:len(offline_start_indices)], 'time'].values,
                'lon': data_sorted.loc[offline_start_indices, 'lon'].values,
                'lat': data_sorted.loc[offline_start_indices, 'lat'].values,
                'next_lon': data_sorted.loc[offline_end_indices[:len(offline_start_indices)], 'lon'].values,
                'next_lat': data_sorted.loc[offline_end_indices[:len(offline_start_indices)], 'lat'].values,
            })
            
            # 计算离线时长（秒）
            offline_periods['duration'] = (offline_periods['end_time'] - offline_periods['start_time']).dt.total_seconds()
            
            # 筛选离线时长在30分钟到4小时之间的记录
            offline_periods = offline_periods[(offline_periods['duration'] >= 1800) & (offline_periods['duration'] <= 14400)]
            
            if offline_periods.empty:
                print(f"车辆ID {target_vehicle_id} 没有符合时长条件的离线期间，跳过离线充电事件识别")
            else:
                ## 位移小于1000m
                # 计算离线前后的位移距离
                offline_periods['displacement'] = tbd.getdistance(
                    offline_periods['lon'], offline_periods['lat'],
                    offline_periods['next_lon'], offline_periods['next_lat']
                )
                
                # 筛选位移小于1000米的记录 
                potential_charge = offline_periods[offline_periods['displacement'] <= 1000].copy()
            
                if len(potential_charge) == 0 or potential_charge.empty:
                    print(f"车辆ID {target_vehicle_id} 没有符合条件的潜在充电事件，跳过离线充电事件识别")
                else:
                    ## 距离最近的充电站200米以内
                    
                    # 计算离线前后点到最近充电站的距离
                    # 离线前点到充电站的距离
                    potential_charge = tbd.ckdnearest(
                        potential_charge, charge_station,
                        Aname=['lon', 'lat'],
                        Bname=['station_lon', 'station_lat']
                    )
                    potential_charge.rename(columns={'dist': 'dist_off'}, inplace=True)
                    
                    # 离线后点到充电站的距离
                    temp_df = pd.DataFrame({
                        'lon': potential_charge['next_lon'],
                        'lat': potential_charge['next_lat']
                    })
                    
                    temp_df = tbd.ckdnearest(
                        temp_df, charge_station,
                        Aname=['lon', 'lat'],
                        Bname=['station_lon', 'station_lat']
                    )
                    
                    potential_charge['dist_on'] = temp_df['dist']
                    potential_charge['station_lon_on'] = temp_df['station_lon']
                    potential_charge['station_lat_on'] = temp_df['station_lat']
                    
                    # 筛选离线前或离线后点到充电站距离小于200米的记录
                    potential_charge = potential_charge[(potential_charge['dist_off'] <= 200) | (potential_charge['dist_on'] <= 200)]
                    
                    if potential_charge.empty:
                        print(f"车辆ID {target_vehicle_id} 没有距离充电站200米以内的离线事件，跳过离线充电事件识别")
                    else:
                        # 标记为离线充电事件
                        potential_charge['ischarge'] = 1
                        potential_charge['charge_type'] = 'offline'
                        
                        # 添加到充电事件列表
                        for _, row in potential_charge.iterrows():
                            charge_event = {
                                'vehicle_id': row['id'],
                                'start_time': row['start_time'],
                                'end_time': row['end_time'],
                                'duration': row['duration'],
                                'lon': row['lon'],
                                'lat': row['lat'],
                                'station_distance': min(row['dist_off'], row['dist_on']),
                                'charge_type': 'offline'
                            }
                            all_charge_events.append(charge_event)
                        
                        # 更新车辆充电次数统计
                        if target_vehicle_id in vehicle_charge_count:
                            vehicle_charge_count[target_vehicle_id]['offline'] = len(potential_charge)
                        else:
                            vehicle_charge_count[target_vehicle_id] = {'online': 0, 'offline': len(potential_charge)}
                        
                        print(f"车辆ID {target_vehicle_id} 识别到 {len(potential_charge)} 个离线充电事件")
                   
end = time.perf_counter()    # 结束时间
print(f"运行时间: {end - start:.2f} 秒")

100%|██████████| 288/288 [00:47<00:00,  6.10it/s]


车辆ID 粤BD00205 没有距离充电站200米以内的停车点，跳过在线充电事件识别
车辆ID 粤BD00205 没有符合时长条件的离线期间，跳过离线充电事件识别
运行时间: 47.33 秒


In [108]:
potential_online_charge

,id,stime,LONCOL,LATCOL,etime,lon,lat,duration,stayid,index,station_lon,station_lat,dist


In [109]:
# 粤BDE5112 
# 1:31:24 - 2:09:00 accuracy=1000
# 1:31:24 - 2:08:15 accuracy=500